# Assignment 2: Two-Player-Zero-Sum Games

## Logistics

You must submit your agent to [Coursemology](https://coursemology.org/courses/3233/assessments/90331).

<span style="color: red">**Note:** Only one submission is permitted per group. If multiple submissions are received from the same group, the last one will be considered. Additionally, any late submissions will result in a penalty for the entire group.</span>

### Grading

This assignment is autograded.

**Marks:**

- Task 1: 3 marks, 1 mark for each sub-task
- Task 2: 2 marks, 1 mark for each sub-task
- Task 3: 6 marks, 1 mark for each sub-task
- Task 4: 4 marks, 1 mark for each sub-task

### Notes

- Only press “Finalise Submission” once you’ve completed all the tasks.
- The assignment won’t be auto-submitted on the deadline.

# The Tian Ji's Horse Racing Game
This question is an extension of a classic [story](https://en.wikipedia.org/wiki/Tian_Ji#The_game) originating from the Warring States period in Chinese history, and is closely related to the Colonel Blotto game that we briefly discussed in class. Both are classic problems in game theory. We will use this opportunity to implement some of the two-player zero-sum game solvers covered in class.

Our modified game setting is as such. There are $n > 1$ horse races to be held. Each of them is worth $(v_1,\dots, v_n) \in \mathbb{R}^n_+$ if won (and $-v_1,\dots,-v_n$ if lost). There are two players, each owns $n$ horses which they will send to the $q$ races. Each horse participates in exactly one race; which race it is assigned to is it's owner's decision. The horses have different speeds, $(\alpha_1,\dots,\alpha_n) \in \mathbb{R}^n_+$ for horses belonging to the first player, and $(\beta_1,\dots,\beta_n) \in \mathbb{R}^n_+$ for the second. For the $j$-th race, the faster horse will win the race, and the winner's owner will obtain a *race value* of $v_j$, while the loser obtains a race value of $-v_j$. If there are ties, both players get a race value of $0$ for that race. The *cumulative race value* is the *sum* of the race values obtained over all $n$ races. This will be the player's total utility.

Observe that this is a two-player zero-sum game where the actions are all possible allocations (i.e., permutations) of horses to races. Therefore, in a naive represenation there are $n!$ actions per player. Each action can be representated by a permutation of $\{ 1,\dots,n \}$. Here, we denote $[\sigma_1,\sigma_2,\sigma_3]$ as the permutation where horse $\sigma_1$ is assigned to race 1, $\sigma_2$ to race 2, etc. 

Remark: The original story assumes that ties were broken in favour of one of the players. We do *not* adopt that position here.

Remark: For simplicity, we have assumed that the performance of horses is independent of their opponent.

Remark: There are much more efficient methods to represent the strategy space that will not involve an exponential blowup in the game size, but these are beyond the scope of the class. Students who are interested may consult me or any reasonable AI model for such an approach.

## Example
Let $n = 3$, and let $\alpha_1=1, \alpha_2=2, \alpha_3=3$, $\beta_1=0.5,\beta_2=1.5,\beta_3=2.5$. Observe that $\alpha_i < \beta_i$ for all horses $i$, so clearly Player 1 has objectively superior horses. The race values are $v_1=10,v_2=15,v_3=20$. 

Suppose Player 1 decides to play greedily such that the fastest horse is assigned to the most valuable track. This is represented by the action $[1,2,3]$, i.e., horse $i$ is assigned to race $i$. Unfortunately, this is extremely exploitable, since Player 2 can play action $[2,3,1]$; this wins races 1 and 2, losing 3, netting Player 2 a utility of $10+15-20=5$. 

Therefore, despite Player 2 having weaker horses, it can still enjoy a strictly positive reward *if Player 1 is not careful in it's play*. Here, Player 2 exploits Player 1 by sacrificing a weak horse to race 3. This is the key takeaway from this classic story.

# Our Goal
Recall from class that in a two-player zero-sum game, the *Nash Equilibrium* is one that maximizes a Player's minimum utility, regardless of what the other player does. This will typically involve randomizing across actions in a clever way to hedge against all possible opponent actions. We will investigate how to compute the equilibrium in this exercise.

We will use the race times obtained from the top 6 finalists in the Men's 100 metre Summer Olympics for the years 2020 and 2016. This is used to compute their speeds. Note that it is only their *relative speeds* (faster, equal, or slower) that matter. The races are worth $v=[1,2,3,4,5,6]$ respectively.

In [15]:
import numpy as np
import itertools
import matplotlib
from matplotlib import pyplot as plt

N = 6

# Times for Player 1 and Player 2
T1 = [9.80, 9.84, 9.89, 9.93, 9.95, 9.98]  # 2020
T2 = [9.81, 9.89, 9.91, 9.93, 9.94, 9.96]  # 2016

# Speeds for Player 1 and Player 2
S1 = 1/np.array(T1) # alpha
S2 = 1/np.array(T2) # beta

# Value of races
V = np.array([1.,2.,3.,4.,5.,6.])

In [ ]:
# Player actions
actions = list(itertools.permutations(list(range(N))))
print(len(actions))

# Part 1: Constructing the Payoff Matrix

The following code implements a function to generate the payoff matrix given inputs $\alpha, \beta, v$. The payoff matrix $A$ is indexed such that Player 1 gets a reward of $A_{ij}$ if Player 1 plays action $i$ and Player 2 action $j$.

In [18]:
def compute_payoff_matrix(S1, S2, V, actions):
  assert len(S1)==len(S2)
  n = len(S1)
  A = np.zeros([len(actions), len(actions)])
  for i in range(len(actions)):
    for j in range(len(actions)):
      accum_payoff = 0.0
      for k in range(n): # race id
        if S1[actions[i][k]]>S2[actions[j][k]]:
          accum_payoff += V[k]
        elif S1[actions[i][k]]<S2[actions[j][k]]:
          accum_payoff -= V[k]
      A[i,j] = accum_payoff
  return A

# Part 2: Computing Payoffs and the Best Response 

In [19]:
def compute_payoff(A, x, y):
  # Return the utility/reward from *Player 1's* perspective
  return np.dot(x, np.dot(A, y))

## Task 1: Compute Best Response (Coursemology Q1)

Given a (possibly random) strategy $y \in \Delta_n$ of Player 2, compute a best response by Player 1. Any such strategy should do. This function should return a tuple containing (i) a numpy array containing a (possibly random) strategy as the best response, and (ii) the utility obtained under the best response. 
Note: if the best response set contains more than one element, choose *any* one. 

> **Coursemology submission (Q1):** Copy your implementation of `compute_best_response` below and paste it into the Coursemology Q1 answer box. The following are available: `compute_payoff`, `np` (`numpy`), and standard Python builtins.

In [ ]:
def compute_best_response(A, y):
    """Compute a best response strategy against opponent's mixed strategy y.

    Args:
        A: payoff matrix of shape (n1, n2)
        y: opponent's mixed strategy, a probability vector of length n2

    Returns:
        A probability vector representing the best response strategy. 
    """
    ## Note: if the best response set contains more than one element, choose *any* one. 
    
    # YOUR CODE STARTS HERE
    raise NotImplementedError()
    # YOUR CODE ENDS HERE


## Task 2: Computing Saddle-point Residual (Coursemology Q2)
Using the best-response function in the previous section, compute the saddle point residual given strategies $x, y \in \Delta_n$.

> **Coursemology submission (Q2):** Copy your implementation of `compute_saddle_point_residual` below and paste it into the Coursemology Q2 answer box. The following are available: `compute_payoff`, `compute_best_response`, `np`.

In [ ]:
def compute_saddle_point_residual(A, x, y):
    """Compute the saddle point residual for strategy profile (x, y).

    The saddle point residual measures how far (x, y) is from a Nash equilibrium.
    It equals zero if and only if (x, y) is a Nash equilibrium.

    Args:
        A: payoff matrix of shape (n1, n2)
        x: Player 1's mixed strategy, a probability vector of length n1
        y: Player 2's mixed strategy, a probability vector of length n2

    Returns:
        A non-negative scalar representing the saddle point residual.
    """
    ## Task 1: Compute the saddle point residual for any strategy profile (x,y). 

    # YOUR CODE STARTS HERE
    raise NotImplementedError()
    # YOUR CODE ENDS HERE


# Part 3: Solving the Game

- We will use the Regret Matching (RM) algorithm to solve for Nash equilibria of the game. 
- The RM algorithm is already scafolded in a later block in function `compute_nash_equilibrium`. 
- We only need to implement two functions: `get_strategy_from_regret` (Task 3) and `update_regret` (Task 4). 

## Task 3: Compute Strategy from Regret (Coursemology Q3)

> **Coursemology submission (Q3):** Copy your implementation of `get_strategy_from_regret` below and paste it into the Coursemology Q3 answer box. The following are available: `np`.

In [ ]:
def get_strategy_from_regret(regret):
    """Convert cumulative regret vector to a mixed strategy using regret matching.

    If all regrets are non-positive, return a uniform distribution.
    Otherwise, normalize the positive regrets to form a probability distribution.

    Args:
        regret: numpy array of cumulative regrets of length n

    Returns:
        A probability vector (numpy array) of length n that sums to 1.
    """
    # YOUR CODE STARTS HERE
    raise NotImplementedError()
    # YOUR CODE ENDS HERE

## Task 4: Update Regrets (Coursemology Q4)

> **Coursemology submission (Q4):** Copy your implementation of `update_regrets` below and paste it into the Coursemology Q4 answer box. The following are available: `np`.

In [ ]:
def update_regrets(A, x, y, regret1, regret2):
    """Update cumulative regrets for both players after one round.

    For each player, the regret for action i is increased by the difference
    between what they would have received by playing action i and what they
    actually received.

    Args:
        A: payoff matrix of shape (n1, n2)
        x: Player 1's current mixed strategy, probability vector of length n1
        y: Player 2's current mixed strategy, probability vector of length n2
        regret1: Player 1's current cumulative regret vector, length n1
        regret2: Player 2's current cumulative regret vector, length n2

    Returns:
        Tuple (new_regret1, new_regret2) of updated regret vectors.
    """
    # YOUR CODE STARTS HERE
    raise NotImplementedError()
    # YOUR CODE ENDS HERE


## Using RM to solve for NE

In [ ]:
def compute_nash_equilibrium(A, max_iter=10000,log_freq=None):
    n1,n2 = A.shape # Number of actions per player

    regret1, regret2 = np.zeros(n1), np.zeros(n2)
    sum_strat1, sum_strat2 = np.zeros(n1), np.zeros(n2)
    spr_log = []

    for t in range(max_iter):
        x = get_strategy_from_regret(regret1)
        y = get_strategy_from_regret(regret2)

        ## Accumulate strategies (used to compute average strategy later)
        sum_strat1 += x
        sum_strat2 += y
        
        regret1, regret2 = update_regrets(A, x, y, regret1, regret2)
        
        ## LOGGING
        if log_freq is not None and t % log_freq == 0:
            x_avg, y_avg = sum_strat1 / (t+1), sum_strat2 / (t+1)
            spr_log.append(compute_saddle_point_residual(A, x_avg, y_avg))

    if log_freq is not None:
        # Plot SPR
        plt.plot(np.array(range(len(spr_log)))*log_freq, spr_log)
        plt.yscale('log')
        plt.show()
    
    x_avg, y_avg = sum_strat1 / (t+1), sum_strat2 / (t+1)
    return x_avg, y_avg 


# Piecing things together, run and verify your implementations

In [ ]:
A = compute_payoff_matrix(S1, S2, V, actions)

print(A)

x, y = compute_nash_equilibrium(A, max_iter=5_000,log_freq=20)


print(x)

# Verify that (x,y) is close to a NE
tol = 2e-1
print('SPR is:', compute_saddle_point_residual(A,x,y))
assert compute_saddle_point_residual(A,x,y) < tol